# T07: Malli A (baseline-CNN), ensimmäinen ajo

Tämä notebook ajaa vain koodia, jonka logiikka on `src/`:ssä (`fold_arrays`, `build_a`, `train_model`). Sisältö:

1. Data: log-mel-välimuisti (T04) ja ajon k = 1 jako
2. Malli A: rakenne ja parametrien määrä
3. Koulutus foldilla k = 1 ja oppimiskäyrät
4. Validointi kaikilla viidellä ajolla ja `lr_patience`-kokeilu

**Vain opetus- ja validointidata.** Testifoldi heitetään pois (`_`), eikä sitä arvioida. Tiedostoja ei tallenneta (`save_files=False`), koska lopulliset ajot, `models/A_fold{k}.keras`, `results/history/A_fold{k}.csv` ja `results/A_fold{k}.json` tehdään T13:ssa viidellä foldilla.

In [ ]:
import sys
from pathlib import Path

# Repon juuri polkuun, jotta `src` löytyy (notebook ajetaan kansiosta notebooks/).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data import build_cache, fold_arrays, load_metadata
from src.models import build_a
from src.train import set_seed, train_model, validate_folds

# Colabissa: DATA_DIR = "/content/drive/MyDrive/Elainaanet-CNN/data"
DATA_DIR = ROOT / "data"
K = 1

## 1. Data

In [ ]:
df = load_metadata(DATA_DIR)
cache = build_cache(df, Path(DATA_DIR) / "cache")          # (400, 128, 431), T04
(x_tr, y_tr), (x_va, y_va), _ = fold_arrays(df, K, cache)  # testifoldia ei käytetä
print("opetus", x_tr.shape, "validointi", x_va.shape)
print("luokkia per osa:", np.bincount(y_tr).tolist(), np.bincount(y_va).tolist())

## 2. Malli A

4 × [Conv2D(3×3, relu) → MaxPool(2×2)] → GlobalAveragePooling2D → Dense(64) → Dense(10). Ei BatchNormia, Dropoutia, L2:ta eikä augmentaatiota (ne ovat mallissa B).

In [ ]:
set_seed()                                                 # siemen 42
model = build_a()
model.summary()

## 3. Koulutus (k = 1)

T06:n yhteiset callbackit: EarlyStopping (`val_loss`, `patience=15`, parhaat painot palautetaan) ja ReduceLROnPlateau. Enintään 100 epookkia, eräkoko 32.

In [ ]:
history, train_seconds = train_model(model, (x_tr, y_tr), (x_va, y_va), "A", K,
                                     epochs=100, batch_size=32, save_files=False)
h = pd.DataFrame(history.history)
h.index = h.index + 1                                      # epookit 1, 2, ...
best = int(h["val_loss"].idxmin())
print(f"{len(h)} epookkia, {train_seconds:.0f} s ({train_seconds / len(h):.1f} s/epookki)")

In [ ]:
val_loss, val_acc = model.evaluate(x_va, y_va, verbose=0)  # parhaan epookin painot
pd.DataFrame({"paras epookki": [best],
              "opetustarkkuus": [h.loc[best, "accuracy"]],
              "validointitarkkuus": [val_acc],
              "validointihäviö": [val_loss]}).round(3)

In [ ]:
# Oppimiskäyrät: häviö ja tarkkuus omissa kuvissaan (eri asteikot, ei kahta y-akselia).
BLUE, ORANGE, TEXT, MUTED, GRID = "#257de0", "#f47230", "#191919", "#666666", "#e5e5e5"
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, metric, title in zip(axes, ["loss", "accuracy"], ["Häviö", "Tarkkuus"]):
    for col, color, name in [(metric, BLUE, "opetus"), (f"val_{metric}", ORANGE, "validointi")]:
        ax.plot(h.index, h[col], color=color, linewidth=2, label=name)
        ax.annotate(name, (h.index[-1], h[col].iloc[-1]), xytext=(6, 0),
                    textcoords="offset points", va="center", color=TEXT)
    ax.axvline(best, color=MUTED, linewidth=1, linestyle="--")
    ax.set_title(f"{title}, malli A, k = {K}", color=TEXT, loc="left")
    ax.set_xlabel(f"Epookki (katkoviiva: paras epookki {best})", color=MUTED)
    ax.grid(axis="y", color=GRID, linewidth=1)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=MUTED, length=0)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, loc="upper right", ncol=2, labelcolor=TEXT)
plt.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

## 4. Validointi kaikilla viidellä ajolla

Sama malli ja asetukset jokaisella ajolla k = 1–5, vain opetus- ja validointidata (`validate_folds`, testiosa heitetään pois). Lisäksi kokeillaan ReduceLROnPlateaun kärsivällisyyttä `lr_patience` 5 (T06:n oletus) ja 10: ajossa k = 1 oppimisnopeus puolittui epookista 23 alkaen viisi kertaa, mikä voi pysäyttää oppimisen liian aikaisin. Tulos tallennetaan tiedostoon `results/T07_a_validointi.csv`.

In [ ]:
fold_data = lambda k: fold_arrays(df, k, cache)
runs = []
for lrp in (5, 10):
    r = validate_folds(build_a, fold_data, "A", lr_patience=lrp)
    r.insert(0, "lr_patience", lrp)
    runs.append(r)
val = pd.concat(runs, ignore_index=True)
val.to_csv(ROOT / "results" / "T07_a_validointi.csv", index=False)
val.pivot_table(index="lr_patience", columns="fold", values="val_acc").round(3)

In [ ]:
(val.groupby("lr_patience")
    .agg(val_acc=("val_acc", "mean"), sd=("val_acc", "std"), train_acc=("train_acc", "mean"),
         epochs=("epochs", "mean"), max_epochs=("epochs", "max"))
    .round(3))

## Havainnot

- **Malli oppii, mutta hitaasti ja huonosti yleistyen.** Validointitarkkuus viiden ajon keskiarvona on noin 0,57 (oletus `lr_patience=5`), ja opetustarkkuuskin jää parhaalla epookilla noin 0,84:ään. Oppimiskäyrissä (k = 1) validointi jää opetuksesta jälkeen noin epookista 20 alkaen.
- **`lr_patience=10` ei parantanut selvästi.** Keskiarvo nousi noin 0,03, mutta yksi ajo parani 10 %-yks., yksi heikkeni ja yksi pysyi samana; erojen hajonta ajojen välillä on samaa suuruusluokkaa kuin keskiarvon ero (validointifoldissa 80 leikettä, 1 leike = 1,25 %-yks.). **Päätös: pidetään T06:n oletus (`lr_patience=5`)**, jotta kaikilla malleilla on samat koulutusasetukset ja A pysyy baselinena.
- **PLAN.md:n tavoite ≥ 70 % (testitarkkuus, 5 foldin keskiarvo) näyttää validoinnin perusteella vaikealta.** Kynnys tarkennetaan PLAN.md:n mukaan T13:n jälkeen, ja siitä päättää ryhmä. Mallin B regularisoinnin ja augmentaation pitäisi auttaa juuri tähän.

Luvut ovat validointituloksia (ajettu kehityskoneella, TF 2.22-rc CPU), eivät mallin A testitulos. Testiajot ja `results/A_fold{k}.json` tehdään T13:ssa.